# <h1 align= 'center'>Base Models</h1>

The goal for this analysis is to predict if a review rates the movie positively or negatively. Inside this dataset, there are 25,000 labelled movies reviews for training, 50,000 unlabeled reviews for training, and 25,000 reviews for testing.
<a href="https://imgur.com/FfdEBRz"><img src="https://i.imgur.com/FfdEBRzm.png" title="source: imgur.com" align="right"></a>
- IMDB movie reviews dataset
- http://ai.stanford.edu/~amaas/data/sentiment
- https://www.kaggle.com/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
- Contains 25000 positive and 25000 negative reviews
- Contains at most reviews per movie
- At least 7 stars out of 10 $\rightarrow$ positive (label = 1)
- At most 4 stars out of 10 $\rightarrow$ negative (label = 0)


> **Here, we use basic regression and classification models to establish baseline performance metrics.**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from xgboost.sklearn import XGBClassifier

from scipy.stats import uniform
from pprint import pprint
import warnings
warnings.filterwarnings('ignore')

## <h2> <center>Load preprocessed data</center></h2>

In [ ]:
# for importing data to colab
from google.colab import drive 
drive.mount('/content/drive')

movies = pd.read_csv('/content/drive/My Drive/Colab Notebooks/cleaned_data.csv')
movies.sample(7)

In [ ]:
# read from local dataset
# movies = pd.read_csv('data/imdb_data.csv')
# movies.sample(7)

## <h2> <center>Text feature extraction</center></h2>

In [ ]:
# todo: normalize if overfit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, f_classif


top_k = 15000

tfidf = TfidfVectorizer(
                    ngram_range = (1,2),
                    dtype = 'int32',
                    decode_error = 'replace',
                    analyzer = 'word',
                    min_df = 2,
                    norm = 'l2'
                    )
tfidf_reviews = tfidf.fit_transform(movies['review'])

# Select best 10,000(k) features, with feature importance measured by f_classif
selector = SelectKBest(f_classif, k = min(top_k, tfidf_reviews.shape[1]))
selector.fit(tfidf_reviews, movies['sentiment'])
vect_reviews = selector.transform(tfidf_reviews).astype('float32')

vect_reviews.shape

## <h2> <center>Split the data</center></h2>

In [ ]:
X = vect_reviews.toarray()
y = (np.array(movies['sentiment']))

# Here we split data to training and testing parts
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)
print(f"Train dataset shape: {X_train.shape}, \nTest dataset shape: {X_test.shape}")

In [ ]:
def plot_confusion_matrix(y_true, y_pred, ax, class_names  = ['Positive', 'Negative'], vmax=None,
                           normalized=True, title='Confusion matrix'):
    """
    Helper fuction to generate a clean Confusion Matrix using seaborn library.
    y_true: True labels, y_pred: Model Predictions, class_names: Override if needed
    normalized: True, gives the proportions instead of absolute numbers
    """
    matrix = confusion_matrix(y_true,y_pred)
    if normalized:
        matrix = matrix.astype('float') / matrix.sum(axis=1)[:, np.newaxis]
    annot_kws = {'fontsize':25, 
                'fontstyle': 'italic'}
    sns.heatmap(matrix, vmax=vmax, annot=True,  annot_kws = annot_kws,
                square=True, ax=ax, cbar=False,
                cmap=sns.diverging_palette(20, 250, as_cmap=True), 
                linecolor='black', linewidths=0.5,
                xticklabels=class_names)
    ax.set_title(title, y=1.20, fontsize=16)
    ax.set_ylabel('True labels', fontsize=12)
    ax.set_xlabel('Predicted labels', y=1.10, fontsize=12)
    ax.set_yticklabels(class_names, rotation=0)

## <h2> <center>Modeling the data</center></h2>

### <h2> <center>Base Logistic Regression model</center></h2>

In [ ]:
lr=LogisticRegression(max_iter=500 ,random_state=7, 
                      C = 10, penalty = 'l2') # found from Random Search 
lr.fit(X_train, y_train)

In [ ]:
pred = lr.predict(X_test)
pred_train = lr.predict(X_train)
print("Training set Accuracy: ", accuracy_score(pred_train, y_train)*100)
print("Test set Accuracy: ", accuracy_score(pred, y_test)*100)

In [ ]:
fig, axis1 = plt.subplots(nrows=1, ncols=1)
plot_confusion_matrix(y_test, pred, ax=axis1,
                       title='Confusion matrix (lr)')

In [ ]:
penalty = ['l2', 'l1']
# Create regularization hyperparameter distribution using uniform distribution
C = [0.1, 10, 100]
logistic = LogisticRegression()
# Create hyperparameter options
hyperparameters = dict(C=C, penalty=penalty)
pprint(hyperparameters)
lr_best = GridSearchCV(logistic, hyperparameters, scoring = 'accuracy',
                       cv = 2, verbose = 0, refit=True, return_train_score = True)

In [ ]:
lr_best.fit(X_train, y_train)

In [ ]:
lr_best_penalty = lr_best.best_estimator_.get_params()['penalty']
lr_best_c = lr_best.best_estimator_.get_params()['C']
print(f'Best Penalty: {lr_best_penalty}')
print(f'Best C: {lr_best_c}')

pprint(lr_best.best_params_)

In [ ]:
pred = lr_best.predict(X_test)
pred_train = lr_best.predict(X_train)
print("Training set Accuracy: ", accuracy_score(pred_train, y_train)*100)
print("Test set Accuracy: ", accuracy_score(pred, y_test)*100)

In [ ]:
fig, axis1 = plt.subplots(nrows=1, ncols=1)
plot_confusion_matrix(y_test, pred, ax=axis1,
                       title='Confusion matrix (lr)')

### <h2> <center>SVM Classifier</center></h2>

In [ ]:
svm = SGDClassifier(random_state = 7, penalty = 'l2', loss = 'hinge')

svm.fit(X_train,y_train)

In [ ]:
pred = svm.predict(X_test)
pred_train = svm.predict(X_train)
print("Training set Accuracy: ", accuracy_score(pred_train, y_train)*100)
print("Test set Accuracy: ", accuracy_score(pred, y_test)*100)

In [ ]:
fig, axis1 = plt.subplots(nrows=1, ncols=1)
plot_confusion_matrix(y_test, pred, ax=axis1,
                       title='Confusion matrix (svm)',
                       class_names=['Positive', 'Negative'])

### <h2> <center>Linear SVC</center></h2>

In [ ]:
svc = LinearSVC(random_state = 7)
svc.fit(X_train, y_train)

In [ ]:
pred = svc.predict(X_test)
pred_train = svc.predict(X_train)
print("Training set Accuracy: ", accuracy_score(pred_train, y_train)*100)
print("Test set Accuracy: ", accuracy_score(pred, y_test)*100)

In [ ]:
fig, axis1 = plt.subplots(nrows=1, ncols=1)
plot_confusion_matrix(y_test, pred, ax=axis1,
                       title='Confusion matrix (svc)',
                       class_names=['Positive', 'Negative'])

In [ ]:
losses = ["hinge", "squared_hinge"]
duals = [False, True]
learning_rates = [1e-1, 1, 10]
penalties = ["l1", "l2"]

hyperparam = {
    "C": learning_rates,
    "penalty": penalties,
    "loss": losses,
    "dual": duals
    }
pprint(hyperparam)
svc = LinearSVC()

n_iter = 50
svc_random = RandomizedSearchCV(svc, hyperparam, cv=2, scoring = 'accuracy',
                                verbose=0, refit = True)

In [ ]:
svc_random.fit(X_train, y_train)

In [ ]:
pprint(svc_random.best_params_)

In [ ]:
pred = svc_random.predict(X_test)
pred_train = svc_random.predict(X_train)
print("Training set Accuracy: ", accuracy_score(pred_train, y_train)*100)
print("Test set Accuracy: ", accuracy_score(pred, y_test)*100)

In [ ]:
fig, axis1 = plt.subplots(nrows=1, ncols=1)
plot_confusion_matrix(y_test, pred, ax=axis1,
                       title='Confusion matrix (svc)',
                       class_names=['Positive', 'Negative'])

### <h2> <center>Multinomial Bayes Classifier</center></h2>

In [ ]:
#training the model
mnb = MultinomialNB(alpha = 0.001) # best alpha found from Grid Search
#fitting the svm for bag of words
mnb.fit(X_train, y_train)

In [ ]:
pred = mnb.predict(X_test)
pred_train = mnb.predict(X_train)
print("Training set Accuracy: ", accuracy_score(pred_train, y_train)*100)
print("Test set Accuracy: ", accuracy_score(pred, y_test)*100)

In [ ]:
fig, axis1 = plt.subplots(nrows=1, ncols=1)
plot_confusion_matrix(y_test, pred, ax=axis1,
                       title='Confusion matrix (mnb)')

In [ ]:
alpha =  [0.001, 0.003, 0.1, 0.3, 1]
hyperparam = dict(alpha = alpha)
mnb_best = GridSearchCV(mnb, hyperparam, scoring = 'accuracy', cv = 2, refit = True, verbose = 1, return_train_score = True)
pprint(hyperparam)

In [ ]:
mnb_best.fit(X_train, y_train)

In [ ]:
mnb_best_alpha = mnb_best.best_estimator_.get_params()['alpha']
print(f'Best alpha: {mnb_best_alpha}')

In [ ]:
pred = mnb_best.predict(X_test)
pred_train = mnb_best.predict(X_train)
print("Training set Accuracy: ", accuracy_score(pred_train, y_train)*100)
print("Test set Accuracy: ", accuracy_score(pred, y_test)*100)

In [ ]:
fig, axis1 = plt.subplots(nrows=1, ncols=1)
plot_confusion_matrix(y_test, pred, ax=axis1,
                       title='Confusion matrix (mnb)')

### <h2> <center>Random Forest Classifier</center></h2>

In [ ]:
# takes a lot of time ...
rf = RandomForestClassifier(max_depth = 100, 
                            n_estimators = 500,
                            min_samples_leaf = 5,
                            min_samples_split = 2,
                            max_features = 'sqrt',
                            random_state = 7)  # found from Random Search

rf.fit(X_train, y_train)

In [ ]:
pred = rf.predict(X_test)
pred_train = rf.predict(X_train)
print("Training set Accuracy: ", accuracy_score(pred_train, y_train)*100)
print("Test set Accuracy: ", accuracy_score(pred, y_test)*100)

In [ ]:
fig, axis1 = plt.subplots(nrows=1, ncols=1)
plot_confusion_matrix(y_test, pred, ax=axis1,
                       title='Confusion matrix (Random Forest)',
                       class_names=['Positive', 'Negative'])

In [ ]:
n_estimators = [int(x) for x in np.linspace(start = 100, stop = 1000, num = 3)]
max_features = ['auto', 'sqrt']
max_depth = [int(x) for x in np.linspace(10, 100, num = 3)]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]
bootstrap = [True, False]

hyperparam = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               }
pprint(hyperparam)
rf_temp = RandomForestClassifier()
rf_random = RandomizedSearchCV(rf_temp, hyperparam, cv = 2, n_iter = 10, verbose = 1,
                        refit = False, return_train_score = True)

In [ ]:
# takes a lot of time ...
# rf_random.fit(X_train, y_train)

In [ ]:
# pprint(rf_random.best_params_)

* The models that performed the best Multinomial Naive Bayes and Support Vector Machine
* Random forest seems to be overfitting and is obviously not suited to this task.